In [ ]:

import netket as nk
import netket.experimental as nkx
import numpy as np
import matplotlib.pyplot as plt
from pyscf import gto, scf, fci
import jax
import jax.numpy as jnp
from flax import nnx
#==============================================================================
# 1. 分子定义 + FCI 精确基准（你的代码）
#==============================================================================
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]

mol = gto.M(atom=geometry, basis='STO-3G')
mf = scf.RHF(mol).run(verbose=0)
E_hf = mf.e_tot

# FCI 精确解（4个态）
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0])*27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

# 转为 NetKet 哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

#==============================================================================
# 2. Hilbert 空间 + 采样器（你的代码）
#==============================================================================
n_orbitals = 2
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=n_orbitals,
    s=1/2,
    n_fermions_per_spin=(1, 1)
)

g = nk.graph.Graph(edges=[(0,1), (2,3)])
sampler = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=4, spin_symmetric=True, sweep_size=64
)

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV


In [ ]:
#==============================================================================
# 3. NES-VMC 神经网络模型（复数 FFNN）
#==============================================================================
K = 2
n_spin_orbitals = 4                   
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        out = jnp.log(out)
        return jnp.squeeze(out)

class NESTotalAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals, n_states=K, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_states = n_states
        self.n_spin_orbitals = n_spin_orbitals
        self.single_ansatz_list = [
            SingleStateAnsatz(n_spin_orbitals, hidden_dim, rngs=nnx.Rngs(42+i))
            for i in range(n_states)
        ]

    def __call__(self, x_batch):
        K_state = self.n_states
        M = []
        for i in range(K_state):
            row = [self.single_ansatz_list[j](x_batch[i]) for j in range(K_state)]
            M.append(jnp.array(row))
        M = jnp.stack(M)
        psi_total = jnp.linalg.det(M)
        log_psi_total = jnp.log(psi_total)
        return log_psi_total, M

# 初始化模型
rngs = nnx.Rngs(42)
total_ansatz = NESTotalAnsatz(n_spin_orbitals, n_states=K, hidden_dim=16, rngs=rngs)

#==============================================================================
# 4. 核心：哈密顿作用 + 局部能量矩阵（论文标准）
#==============================================================================
def Ham_psi(ha: nk.operator.DiscreteOperator, single_ansatz, x: jnp.array):
    x_primes, mels = ha.get_conn(x)
    psi_vals = jax.vmap(single_ansatz)(x_primes)
    return jnp.sum(mels * psi_vals)

def Ham_Psi(ha:nk.operator.DiscreteOperator, total_ansatz: NESTotalAnsatz, x_batch):
    K_state = total_ansatz.n_states
    H_mat = []
    for i in range(K_state):
        row = []
        for j in range(K_state):
            v = Ham_psi(ha, total_ansatz.single_ansatz_list[j], x_batch[i])
            row.append(v)
        H_mat.append(row)
    return jnp.array(H_mat, dtype=complex)

def compute_local_energy(ha:nk.operator.DiscreteOperator, total_ansatz: NESTotalAnsatz, x_batch, return_log_psi=False):
    psi, M = total_ansatz(x_batch, return_log_psi=return_log_psi)
    # 🔥 修复 1：对角加载，防止矩阵奇异
    eps = 1e-3
    M = M + eps * jnp.eye(M.shape[0], dtype=M.dtype)
    Hp = Ham_Psi(ha, total_ansatz, x_batch)
    el_mat = jnp.linalg.solve(M, Hp)
    return jnp.trace(el_mat), el_mat

#==============================================================================
# 5. 损失函数 & 训练步骤（JAX 自动求导）
#==============================================================================
@jax.jit
def loss_fn(model_state, samples):
    nnx.update(total_ansatz, model_state)
    total_energy = 0.0 + 0j
    n_samples = samples.shape[0]
    for xb in samples:
        tr_EL, _ = compute_local_energy(ha, total_ansatz, xb)
        total_energy += tr_EL
    avg_energy = total_energy.real / n_samples
    return avg_energy

@jax.jit
def train_step(model_state, samples):
    loss, grads = jax.value_and_grad(loss_fn)(model_state, samples)
    model_state = jax.tree_util.tree_map(lambda p, g: p - 0.01 * g, model_state, grads)
    return model_state, loss



In [4]:
import jax
import jax.numpy as jnp
import netket as nk
from netket.utils.types import Array

# =============================================================================
# 🔥 NES-VMC 专用：费米子 K-构型 联合采样器
# 作用：一次采样 一整组 X = [x¹, x², ..., xᴷ]
# 规则：每次随机更新其中 1 个构型，其余不动 → 标准 single-site update
# =============================================================================
class NESFermionSampler:
    def __init__(self, hi: nk.hilbert.SpinOrbitalFermions, graph, K_states: int):
        self.hi = hi
        self.graph = graph
        self.K = K_states  # 要同时采样的构型数量（你要算几个态就填几）
        self.n_sites = hi.size
        self.n_fermions = hi.n_fermions_per_spin[0]  # 假设自旋上下数量相同（可改）

    # 初始化一整组 K 个费米子构型
    def init_samples(self, key):
        keys = jax.random.split(key, self.K)
        samples = jax.vmap(self._init_single)(keys)
        return samples  # shape: (K, n_sites)

    # 单个费米子构型初始化（保证粒子数守恒）
    def _init_single(self, key):
        key1, key2 = jax.random.split(key)
        # 随机占据，但固定费米子数目
        occ = jnp.zeros(self.n_sites, dtype=jnp.int32)
        idx = jax.random.permutation(key1, self.n_sites)[: self.n_fermions]
        occ = occ.at[idx].set(1)
        return occ

    # 🔥 核心：Metropolis 一步，更新整组 X
    def sample_step(self, log_psi_total, key, X_current):
        # X_current shape = (K, n_sites)  【一整组样本】
        K = self.K

        # 1) 随机选 一个 构型 xᵏ 进行更新
        key, subk = jax.random.split(key)
        k_update = jax.random.randint(subk, (), 0, K)

        # 2) 对选中的 xᵏ 做费米子跃迁（Hop 操作，保证粒子数守恒）
        x_old = X_current[k_update]
        key, x_new = self._fermion_hop(key, x_old)

        # 3) 构造新的整组样本
        X_prop = X_current.at[k_update].set(x_new)

        # 4) 计算 NES 总波函数 logΨ (行列式波函数)
        log_psi_old = log_psi_total(X_current)
        log_psi_new = log_psi_total(X_prop)

        # 5) Metropolis 接受概率
        accept_prob = jnp.exp(2 * (log_psi_new - log_psi_old))
        accept_prob = jnp.minimum(accept_prob, 1.0)

        key, subk = jax.random.split(key)
        accept = jax.random.uniform(subk, ()) < accept_prob

        # 6) 接受/拒绝
        X_new = jax.lax.cond(accept, lambda: X_prop, lambda: X_current)
        return key, X_new, accept

    # ------------------------------
    # 费米子专用：Hop 更新（粒子数守恒）
    # 随机选一个占据态 → 跳到空态
    # ------------------------------
    def _fermion_hop(self, key, x):
        occupied = jnp.where(x == 1)[0]
        empty = jnp.where(x == 0)[0]

        key1, key2 = jax.random.split(key)
        i = jax.random.choice(key1, occupied)
        j = jax.random.choice(key2, empty)

        x_new = x.at[i].set(0)
        x_new = x_new.at[j].set(1)
        return key, x_new

    # 运行采样（生成一批样本）
    def sample(self, log_psi_total, key, n_steps=5000, warmup=1000):
        X = self.init_samples(key)
        accept_sum = 0

        # Warmup
        for _ in range(warmup):
            key, X, _ = self.sample_step(log_psi_total, key, X)

        # 正式采样
        samples = []
        for _ in range(n_steps):
            key, X, acc = self.sample_step(log_psi_total, key, X)
            accept_sum += acc
            samples.append(X)

        samples = jnp.array(samples)
        accept_rate = accept_sum / n_steps
        return samples, accept_rate

In [5]:
nes_sampler = NESFermionSampler(hi=hi,graph=g,K_states=2)

In [ ]:
total_ansatz(x_batch=jnp.array([[0, 1, 0, 1],[1,0,1,0]]))

(Array(0.98442375-2.84166006j, dtype=complex128),
 Array([[-1.34886279+0.90503608j, -0.27693885+1.36189607j],
        [ 0.08809921-1.71079572j,  0.06089442+0.18687087j]],      dtype=complex128))